# ch05 Bonus 12：Gemma3 实现

> 对照官方 `ch05/12_gemma3`
> **参考真实模型**：Google Gemma3（2025）

## 一句话

Gemma3 在 Llama 基础上加 **滑动窗口注意力（SWA）+ QK-Norm + 缩放 embedding**，并用 5:1 的全局/局部注意力交替。

## 相对 Llama 的独特改造

1. **滑动窗口注意力（每 6 层中 5 层用 SWA，1 层全局）**：长文本省算力（见 ch04/06_swa）
2. **QK-Norm**：同 Qwen3
3. **缩放 embedding**：token embedding 乘一个 sqrt(d) 的缩放因子，防止初始值过大
4. **post-norm 残差**：部分层用 post-LN 结构

> Gemma 系列的标志性特点是把 SWA 做成了生产级（配合全局层定期捕捉远距离依赖）。

In [ ]:
import torch
import torch.nn as nn

# 改造 1：缩放 embedding（Gemma 标志性特点）
class ScaledEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
    def forward(self, x):
        # Gemma：embedding 乘 sqrt(emb_dim) 防止初始值过大
        return self.emb(x) * (self.emb.embedding_dim ** 0.5)

se = ScaledEmbedding(1000, 768)
idx = torch.tensor([[1, 2, 3]])
print(f"缩放 embedding: 普通={nn.Embedding(1000,768)(idx)[0,0].abs().mean():.4f}")
print(f"               缩放={se(idx)[0,0].abs().mean():.4f}（约 √768≈27.7 倍）")

# 改造 2：Gemma3 的 5:1 全局/局部注意力模式
print("\nGemma3 注意力模式（每 6 层一组）：")
for i in range(12):
    mode = "全局" if i % 6 == 5 else "滑窗(SWA)"
    print(f"  层 {i:2d}: {mode}")
print("💡 5 层滑窗省算力 + 1 层全局保证远距离信息能传递。")